In [1]:
# --- Cell 1: Smart Offline Install (SAFE & MINIMAL) ---
import sys, os, glob, subprocess

print("⚙️ Smart offline install (minimal + compatible wheels only)...")

FORBIDDEN = ("torch", "torchvision", "torchaudio", "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow")
NEEDED_HINTS = ("segmentation_models_pytorch", "ultralytics", "timm")

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"   # cp311
print("🐍 Python tag:", py_tag)

all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 Found wheels: {len(all_whls)}")

def ok_whl(name: str) -> bool:
    n = name.lower()
    if any(x in n for x in FORBIDDEN):
        return False
    return ("py3-none-any" in n) or (py_tag in n)

picked = []
for p in all_whls:
    n = os.path.basename(p)
    if any(h in n.lower() for h in NEEDED_HINTS) and ok_whl(n):
        picked.append(p)

picked = sorted(set(picked), key=len)
print("✅ Will try install:", len(picked), "wheels")

ok = 0
for whl in picked:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", whl, "--no-deps", "--ignore-installed", "--quiet"])
        ok += 1
    except:
        print("⚠️ Failed:", os.path.basename(whl))

print(f"✅ Installed {ok}/{len(picked)} wheels safely.")

import torch
print("⚡ Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


⚙️ Smart offline install (minimal + compatible wheels only)...
🐍 Python tag: cp311
📦 Found wheels: 95
✅ Will try install: 3 wheels
✅ Installed 3/3 wheels safely.
⚡ Torch: 2.6.0+cu124 | CUDA: True
GPU: Tesla T4


In [2]:
# --- Cell 2: Offline install/import fix + model path resolver (NO internet) ---

import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات الضرورية (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle env may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))

# --------- Resolve model paths (robust) ----------
def find_first(patterns):
    hits = []
    for p in patterns:
        hits += glob.glob(p, recursive=True)
    hits = sorted(hits, key=len)
    return hits[0] if hits else None

# NEW model (EffB3)
NEW_UNET_PATH = find_first([
    "/kaggle/input/**/best_model_effb3_phase9_ddp.pth",
    "/kaggle/input/**/best_model_effb3_phase9_ddp*.pth",
])

# OLD model (Phase10)
OLD_UNET_PATH = find_first([
    "/kaggle/input/**/best_model_phase10.pth",
    "/kaggle/input/**/best_model_phase10*.pth",
])

# YOLO
YOLO_PATH = find_first([
    "/kaggle/input/**/best.pt",
])

print("📌 Resolved paths:")
print(" - NEW_UNET_PATH:", NEW_UNET_PATH)
print(" - OLD_UNET_PATH:", OLD_UNET_PATH)
print(" - YOLO_PATH    :", YOLO_PATH)

print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
📌 Resolved paths:
 - NEW_UNET_PATH: /kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth
 - OLD_UNET_PATH: /kaggle/input/ecg-final-models1/best_model_phase10.pth
 - YOLO_PATH    : /kaggle/input/ecg-final-models1/best.pt
✅ Cell 2 ready.


In [3]:
# --- Cell 2.5: Attach NEW trained model (EffB3) to inference ---
import os, shutil

# عدّل لو عندك مسار مختلف
NEW_BEST_SRC = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"

# إذا كنت مخزنها في Dataset input بدل working ضع المسار هنا:
# NEW_BEST_SRC = "/kaggle/input/<your-dataset>/best_model_effb3_phase9_ddp.pth"

NEW_BEST_DST = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"

if os.path.exists(NEW_BEST_SRC):
    if NEW_BEST_SRC != NEW_BEST_DST:
        shutil.copy(NEW_BEST_SRC, NEW_BEST_DST)
    print("✅ NEW model ready:", NEW_BEST_DST)
else:
    print("❌ NEW model not found at:", NEW_BEST_SRC)

print("Exists:", os.path.exists(NEW_BEST_DST))


✅ NEW model ready: /kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth
Exists: True


In [4]:
# # --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# # إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
# DB_DIR = "physionet_db"
# if not os.path.exists(DB_DIR):
#     os.makedirs(DB_DIR)
#     print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
#     try:
#         # تحميل عينة كافية لضمان التنوع
#         records = wfdb.get_record_list('ptb-xl')
#         selected_records = records[:200] 
#         wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
#         print(f"✅ تم تحميل {len(selected_records)} سجل.")
#     except Exception as e:
#         print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

# class UltimateRenderer:
#     def __init__(self, db_dir):
#         self.db_dir = db_dir
#         self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
#     def get_real_signal(self):
#         """سحب إشارة عشوائية من PTB-XL"""
#         if not self.records:
#             t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
#         try:
#             rec_name = random.choice(self.records)
#             record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
#             lead_idx = random.randint(0, record.shape[1] - 1)
#             signal = record[:, lead_idx]
#             signal = np.nan_to_num(signal)
            
#             # قص عشوائي (Zoom in/out simulation)
#             crop_len = random.randint(1000, 3000)
#             if len(signal) > crop_len:
#                 start = random.randint(0, len(signal) - crop_len)
#                 return signal[start : start+crop_len]
#             return signal
#         except:
#             return np.random.randn(2000)

#     def render_to_memory(self, signal):
#         """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
#         # 3. الشبكة المتغيرة (Distractor)
#         grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
#         grid_alpha = random.uniform(0.3, 0.8)
#         bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
#         fig_h, fig_w = 3, 8
#         dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
#         # --- أ. توليد القناع (Mask Generation) ---
#         fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
#         ax.plot(signal, color='white', linewidth=3.0) 
#         ax.set_ylim(np.min(signal), np.max(signal))
#         ax.axis('off')
#         fig.patch.set_facecolor('black')
        
#         fig.canvas.draw()
#         mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#         mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#         mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
#         _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
#         plt.close(fig)

#         # --- ب. الرسم الأولي (Rendering) ---
#         fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
#         ax.set_facecolor(bg_color)
        
#         # رسم الشبكة
#         ax.minorticks_on()
#         ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
#         ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
#         # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
#         line_color = random.choice(['black', 'blue', '#000033']) 
#         ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
#         ax.axis('off')
#         ax.set_ylim(np.min(signal), np.max(signal))
#         fig.patch.set_facecolor(bg_color)
        
#         fig.canvas.draw()
#         img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#         img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#         # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
#         img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
#         plt.close(fig)
        
#         return img, mask

# print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")
# --- Cell 3: (Disabled in Submission) Renderer/PTB-XL not needed ---
print("✅ Cell 3 disabled: Renderer/PTB-XL not needed for submission.")


✅ Cell 3 disabled: Renderer/PTB-XL not needed for submission.


In [5]:
# --- Cell 22: V40 FINAL (test.csv path map + ImageNet norm + NEW/OLD ensemble + robust DP) ---

import os, gc, csv, glob
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm import tqdm
from collections import OrderedDict
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------------------------
# 0) Paths / Constants
# ---------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV_PATH = f"{IMAGE_DIR}/test.csv"
SAMPLE_PARQUET_PATH = f"{IMAGE_DIR}/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

NEW_UNET_PATH = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"
OLD_UNET_PATH = "/kaggle/input/ecg-final-models1/best_model_phase10.pth"
YOLO_PATH     = "/kaggle/input/ecg-final-models1/best.pt"

print("⚡ Device:", DEVICE)
print("📌 Paths exist:",
      "| NEW:", os.path.exists(NEW_UNET_PATH),
      "| OLD:", os.path.exists(OLD_UNET_PATH),
      "| YOLO:", os.path.exists(YOLO_PATH))

USE_OLD = os.path.exists(OLD_UNET_PATH)
ALPHA_NEW = 0.75  # 75% NEW + 25% OLD (مفيد عادة)

TARGET_H = 256
MAX_W = 2048
YOLO_INF_MAX = 1280
YOLO_CONF = 0.10
MAX_CACHE = 8

DP_MAX_W = 1400
DP_SMOOTH = 0.45

torch.backends.cudnn.benchmark = True

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_TO_IDX = {n:i for i,n in enumerate(LEAD_NAMES)}

# ImageNet normalization (ضروري لـ EfficientNet)
IM_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3,1,1)
IM_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3,1,1)

def norm_chw(x):
    return (x - IM_MEAN) / (IM_STD + 1e-8)

def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

# ---------------------------
# 1) Read template ids + per-patient length
# ---------------------------
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
template_ids = tmpl["id"].astype(str).to_numpy()
del tmpl
gc.collect()

def parse_rid(rid: str):
    parts = rid.rsplit("_", 2)
    if len(parts) != 3:
        return None
    pid_raw, idx_raw, lead = parts
    try:
        idx = int(idx_raw)
    except:
        return None
    return pid_raw, idx, lead

patient_lengths = {}
bad_parse = 0
for rid in template_ids:
    pr = parse_rid(rid)
    if pr is None:
        bad_parse += 1
        continue
    pid_raw, idx, lead = pr
    cur = patient_lengths.get(pid_raw, 0)
    if idx + 1 > cur:
        patient_lengths[pid_raw] = idx + 1

print("✅ Template rows:", len(template_ids))
print("✅ patient_lengths mapped:", len(patient_lengths), "| bad_parse:", bad_parse)

# ---------------------------
# 2) Build pid->image_path from test.csv (CRITICAL)
# ---------------------------
pid_to_path = {}
fs_map = {}

if os.path.exists(TEST_CSV_PATH):
    tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
    low = {c.lower(): c for c in tdf.columns}

    # id column
    id_c = None
    for k in ["id", "record_id", "patient_id"]:
        if k in low:
            id_c = low[k]
            break
    if id_c is None:
        # fallback any col contains 'id'
        id_c = next((low[c] for c in low if "id" in c), None)

    # fs column
    fs_c = next((low[c] for c in low if c in ("fs","sampling_rate") or "fs" in c or "sampl" in c), None)

    # image/path column (الأهم)
    path_c = None
    for k in ["path", "image_path", "filepath", "file_path", "filename", "image", "img"]:
        if k in low:
            path_c = low[k]
            break
    if path_c is None:
        path_c = next((low[c] for c in low if "path" in c or "file" in c or "image" in c), None)

    if id_c and path_c:
        for pid, p in zip(tdf[id_c].astype(str), tdf[path_c].astype(str)):
            pid2 = clean_pid(pid)
            p2 = str(p)
            # إذا مسار نسبي، اربطه بالداتا سيت
            if not p2.startswith("/"):
                p2 = os.path.join(IMAGE_DIR, p2)
            pid_to_path[pid2] = p2

    if id_c and fs_c:
        fs_map = dict(zip(tdf[id_c].astype(str).map(clean_pid), tdf[fs_c].astype(str)))

    print("✅ test.csv columns:", list(tdf.columns))
    print("✅ pid_to_path:", len(pid_to_path), "| fs_map:", len(fs_map))
else:
    print("⚠️ test.csv not found (unexpected).")

# fallback glob indexing (للأمان)
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print("✅ glob indexed images:", len(path_map))

def get_image_path(pid_raw):
    pid = clean_pid(pid_raw)
    # 1) from test.csv mapping
    p = pid_to_path.get(pid, None)
    if p and os.path.exists(p):
        return p
    # 2) from glob basename
    if pid in path_map:
        return path_map[pid]
    try:
        k = str(int(float(pid)))
        return path_map.get(k, None)
    except:
        return None

def get_fs(pid_raw, default=500.0):
    pid = clean_pid(pid_raw)
    v = fs_map.get(pid, None)
    try:
        return float(v) if v is not None else float(default)
    except:
        return float(default)

def safe_read_rgb(path):
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ---------------------------
# 3) Load YOLO + class mapping
# ---------------------------
yolo_model = None
CLASS_TO_LEAD_IDX = {}

def _normalize_lead_name(x: str) -> str:
    s = str(x).strip()
    s = s.replace("Lead","").replace("lead","").replace("_","").replace("-","").replace(" ","")
    s = s.replace("AVR","aVR").replace("AVL","aVL").replace("AVF","aVF")
    s = s.replace("AVr","aVR").replace("AVl","aVL").replace("AVf","aVF")
    return s

if os.path.exists(YOLO_PATH):
    yolo_model = YOLO(YOLO_PATH)
    names = getattr(yolo_model, "names", None)
    if isinstance(names, dict):
        items = list(names.items())
    elif isinstance(names, list):
        items = list(enumerate(names))
    else:
        items = []
    for cid, cname in items:
        n = _normalize_lead_name(cname)
        if n in LEAD_TO_IDX:
            CLASS_TO_LEAD_IDX[int(cid)] = LEAD_TO_IDX[n]
    for i in range(12):
        CLASS_TO_LEAD_IDX.setdefault(i, i)
    print("✅ YOLO loaded:", YOLO_PATH)
else:
    print("⚠️ YOLO missing -> outputs may be zeros.")

# ---------------------------
# 4) Load UNet NEW + OLD (robust)
# ---------------------------
def load_state(path):
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict) and "state_dict" in obj:
        return obj["state_dict"]
    if isinstance(obj, dict) and "model" in obj:
        return obj["model"]
    return obj

def build_unet_effb3():
    return smp.Unet(
        encoder_name="efficientnet-b3",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        decoder_attention_type="scse"
    )

def build_unet_resnet50():
    return smp.Unet(
        encoder_name="resnet50",
        encoder_weights=None,
        in_channels=3,
        classes=1,
        decoder_attention_type="scse"
    )

new_model = build_unet_effb3()
sd_new = load_state(NEW_UNET_PATH)
new_model.load_state_dict(sd_new, strict=False)   # EMA-safe
new_model.to(DEVICE).eval()
print("✅ NEW loaded")

old_model = None
if USE_OLD:
    # جرب ResNet50 أولاً (غالبًا phase10 كان resnet50)
    try:
        old_model = build_unet_resnet50()
        sd_old = load_state(OLD_UNET_PATH)
        old_model.load_state_dict(sd_old, strict=False)
        old_model.to(DEVICE).eval()
        print("✅ OLD loaded (resnet50)")
    except:
        try:
            old_model = build_unet_effb3()
            sd_old = load_state(OLD_UNET_PATH)
            old_model.load_state_dict(sd_old, strict=False)
            old_model.to(DEVICE).eval()
            print("✅ OLD loaded (effb3)")
        except:
            old_model = None
            print("⚠️ OLD load failed -> using NEW only")

# ---------------------------
# 5) Helpers: grid remove + calibration + filters
# ---------------------------
def preprocess_remove_red_grid(img_rgb):
    try:
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        m = (cv2.inRange(hsv, (0,50,50), (10,255,255)) |
             cv2.inRange(hsv, (170,50,50), (180,255,255)))
        out = img_rgb.copy()
        out[m > 0] = (255,255,255)
        return out
    except:
        return img_rgb

def estimate_grid_spacing_px(img_rgb):
    try:
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        if gray.size == 0 or np.std(gray) < 3:
            return None
        ex = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        ey = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(ex), axis=0)
        sy = np.sum(np.abs(ey), axis=1)
        px, _ = find_peaks(sx, distance=10, prominence=np.percentile(sx, 75))
        py, _ = find_peaks(sy, distance=10, prominence=np.percentile(sy, 75))
        diffs = []
        if len(px) > 5:
            dx = np.diff(px); diffs.extend(dx[(dx>8) & (dx<140)])
        if len(py) > 5:
            dy = np.diff(py); diffs.extend(dy[(dy>8) & (dy<140)])
        if len(diffs) < 6:
            return None
        return float(np.median(np.array(diffs)))
    except:
        return None

def choose_ppmv(local_grid_px, resize_scale, raw_trace_px):
    if local_grid_px is None or (not np.isfinite(local_grid_px)) or local_grid_px < 5:
        return 250.0
    ppmvA = (local_grid_px * 10.0) * resize_scale  # assume 1mm
    ppmvB = (local_grid_px * 2.0)  * resize_scale  # assume 5mm

    def score(ppmv):
        if ppmv <= 0: return 1e9
        sig = (raw_trace_px - np.median(raw_trace_px)) / ppmv
        sig = np.nan_to_num(sig)
        p2p = float(np.percentile(sig,99) - np.percentile(sig,1))
        if p2p <= 0: return 1e9
        if p2p < 0.2: return 1000 + (0.2-p2p)*1000
        if p2p > 8.0: return 1000 + (p2p-8.0)*300
        return abs(p2p - 2.5)

    sA, sB = score(ppmvA), score(ppmvB)
    ppmv = ppmvA if sA <= sB else ppmvB
    if not np.isfinite(ppmv) or ppmv < 10:
        return 250.0
    return float(ppmv)

def apply_high_pass(data, cutoff=0.5, fs=500.0, order=3):
    try:
        if len(data) < order * 3:
            return data
        nyq = 0.5 * fs
        if nyq <= 0:
            return data
        wn = cutoff / nyq
        if not (0 < wn < 1):
            return data
        b, a = butter(order, wn, btype="high")
        return filtfilt(b, a, data)
    except:
        return data

# ---------------------------
# 6) Viterbi/DP + fill missing columns
# ---------------------------
def viterbi_dp(prob_map, smooth=0.45):
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6).astype(np.float32)
    dp = np.zeros_like(cost, dtype=np.float32)
    parent = np.zeros((H, W), dtype=np.int16)

    dp[:,0] = cost[:,0]
    parent[:,0] = np.arange(H, dtype=np.int16)

    for t in range(1, W):
        prev = dp[:, t-1]
        c_m1 = np.roll(prev, 1);  c_m1[0]  = 1e9
        c_0  = prev
        c_p1 = np.roll(prev,-1);  c_p1[-1] = 1e9

        stacked = np.stack([c_m1 + smooth, c_0, c_p1 + smooth], axis=0)
        which = np.argmin(stacked, axis=0).astype(np.int16)
        best_prev = stacked[which, np.arange(H)]
        dp[:,t] = cost[:,t] + best_prev
        parent[:,t] = np.clip(np.arange(H, dtype=np.int16) + (which - 1), 0, H-1)

    path = np.zeros(W, dtype=np.int32)
    path[-1] = int(np.argmin(dp[:,-1]))
    for t in range(W-2, -1, -1):
        path[t] = int(parent[path[t+1], t+1])
    return path

def extract_trace(prob):
    H, W = prob.shape
    col_max = prob.max(axis=0)
    good = col_max > 0.10

    if W <= DP_MAX_W:
        path = viterbi_dp(prob, smooth=DP_SMOOTH).astype(np.float32)
    else:
        path = np.argmax(prob, axis=0).astype(np.float32)
        win = 21 if W >= 21 else (W if W%2==1 else max(5, W-1))
        if win >= 5:
            path = savgol_filter(path, win, 2)

    if good.sum() >= 10 and (~good).any():
        xs = np.arange(W)
        path2 = path.copy()
        path2[~good] = np.interp(xs[~good], xs[good], path[good])
        path = path2

    raw_px = (H - path).astype(np.float32)
    return raw_px

# ---------------------------
# 7) YOLO crops
# ---------------------------
def get_crops_yolo_mapped(img_rgb):
    crops = [None]*12
    if yolo_model is None:
        return crops

    h0, w0 = img_rgb.shape[:2]
    scale = YOLO_INF_MAX / float(max(h0, w0))
    if scale < 1.0:
        w1, h1 = max(32, int(w0*scale)), max(32, int(h0*scale))
        img_inf = cv2.resize(img_rgb, (w1, h1))
    else:
        img_inf = img_rgb
        scale = 1.0

    best = {}
    res = yolo_model.predict(img_inf, conf=YOLO_CONF, verbose=False, device=DEVICE)
    if res and res[0].boxes is not None and len(res[0].boxes) > 0:
        boxes = res[0].boxes.data.detach().cpu().numpy()
        for b in boxes:
            x1,y1,x2,y2,conf,cls_id = b[:6]
            cls_id = int(cls_id); conf=float(conf)
            li = CLASS_TO_LEAD_IDX.get(cls_id, cls_id)
            if not (0 <= li < 12):
                continue
            x1o,y1o,x2o,y2o = x1/scale, y1/scale, x2/scale, y2/scale
            x1o,y1o = max(0,int(x1o)), max(0,int(y1o))
            x2o,y2o = min(w0,int(x2o)), min(h0,int(y2o))
            if x2o <= x1o+10 or y2o <= y1o+10:
                continue
            prev = best.get(li)
            if prev is None or conf > prev[0]:
                best[li] = (conf, (x1o,y1o,x2o,y2o))

    for li, (conf, (x1o,y1o,x2o,y2o)) in best.items():
        crops[li] = img_rgb[y1o:y2o, x1o:x2o]
    return crops

# ---------------------------
# 8) Batch predict with ImageNet norm + flip TTA + NEW/OLD ensemble
# ---------------------------
def prepare_batch(crops_rgb):
    tens_list, scales, widths = [], [], []
    for c in crops_rgb:
        h,w = c.shape[:2]
        if h < 5 or w < 5:
            continue
        sc = TARGET_H / float(h)
        nw = int(max(1, w*sc))
        if nw > MAX_W:
            nw = MAX_W
            sc = nw / float(w)

        rz = cv2.resize(c, (nw, TARGET_H))
        t = torch.from_numpy(rz).permute(2,0,1).float() / 255.0
        arr = t.numpy()
        arr = norm_chw(arr)
        t = torch.from_numpy(arr)

        tens_list.append(t)
        scales.append(sc)
        widths.append(nw)

    if not tens_list:
        return None, None, None

    max_w = int(np.ceil(max(widths)/32.0)*32)
    max_w = min(max_w, MAX_W)

    batch = torch.zeros((len(tens_list),3,TARGET_H,max_w), dtype=torch.float32)
    for i,t in enumerate(tens_list):
        w_i = min(t.shape[2], max_w)
        batch[i,:,:,:w_i] = t[:,:,:w_i]
    return batch.to(DEVICE), scales, widths

@torch.inference_mode()
def run_model(m, x):
    if DEVICE == "cuda":
        with torch.amp.autocast("cuda", enabled=True):
            p1 = torch.sigmoid(m(x))
            p2 = torch.sigmoid(m(torch.flip(x, [3])))
            return (p1 + torch.flip(p2, [3])) / 2.0
    else:
        p1 = torch.sigmoid(m(x))
        p2 = torch.sigmoid(m(torch.flip(x, [3])))
        return (p1 + torch.flip(p2, [3])) / 2.0

def predict_probmaps(crops_rgb):
    batch, scales, widths = prepare_batch(crops_rgb)
    if batch is None:
        return [], []

    p_new = run_model(new_model, batch)
    if old_model is not None:
        p_old = run_model(old_model, batch)
        preds = ALPHA_NEW * p_new + (1.0 - ALPHA_NEW) * p_old
    else:
        preds = p_new

    preds = preds.detach().float().cpu().numpy()
    prob_maps = []
    for i in range(preds.shape[0]):
        w_i = widths[i]
        prob_maps.append(preds[i,0,:,:w_i].astype(np.float32))
    return prob_maps, scales

# ---------------------------
# 9) Patient compute
# ---------------------------
patient_cache = OrderedDict()
did_debug_once = False

def compute_patient_leads(pid_raw, target_len):
    out = np.zeros((12, target_len), dtype=np.float32)

    path = get_image_path(pid_raw)
    if not path:
        return out

    img = safe_read_rgb(path)
    if img is None:
        return out

    fs = get_fs(pid_raw, default=500.0)

    crops = get_crops_yolo_mapped(img)
    detected = [(i,c) for i,c in enumerate(crops) if c is not None and c.size > 200]
    if not detected:
        return out

    lead_indices = [i for i,_ in detected]
    crops_rgb = [preprocess_remove_red_grid(c) for _,c in detected]

    prob_maps, scales = predict_probmaps(crops_rgb)
    if not prob_maps:
        return out

    global_grid = estimate_grid_spacing_px(img)

    for j in range(len(prob_maps)):
        lead_idx = lead_indices[j]
        prob = prob_maps[j]
        sc = scales[j]

        raw_px = extract_trace(prob)

        local_grid = estimate_grid_spacing_px(detected[j][1])
        if local_grid is None:
            local_grid = global_grid

        ppmv = choose_ppmv(local_grid, sc, raw_px)
        sig = (raw_px - np.median(raw_px)) / ppmv
        sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)

        if len(sig) >= 11:
            sig = savgol_filter(sig, 11, 3)
        if len(sig) >= 30:
            sig = apply_high_pass(sig, cutoff=0.5, fs=fs, order=3)

        sig = np.clip(sig, -8.0, 8.0)
        out[lead_idx] = resample(sig, target_len).astype(np.float32)

    return out

# ---------------------------
# 10) Write submission
# ---------------------------
print("🚀 Writing submission.csv (V40 FINAL)...")

with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for rid in tqdm(template_ids, desc="Processing Rows"):
        pr = parse_rid(rid)
        if pr is None:
            writer.writerow([rid, "0.0000"])
            continue

        pid_raw, idx, lead_name = pr
        t_len = patient_lengths.get(pid_raw, 5000)
        if t_len <= 0 or t_len > 20000:
            t_len = 5000

        if pid_raw in patient_cache:
            mtx = patient_cache[pid_raw]
            patient_cache.move_to_end(pid_raw)
        else:
            mtx = compute_patient_leads(pid_raw, t_len)
            patient_cache[pid_raw] = mtx
            if len(patient_cache) > MAX_CACHE:
                patient_cache.popitem(last=False)

        lead_idx = LEAD_TO_IDX.get(lead_name, 0)

        val = 0.0
        if 0 <= idx < mtx.shape[1]:
            v = float(mtx[lead_idx][idx])
            if np.isfinite(v):
                val = v

        writer.writerow([rid, f"{val:.4f}"])

del patient_cache
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("✅ Done. submission.csv ready.")


⚡ Device: cuda
📌 Paths exist: | NEW: True | OLD: True | YOLO: True
✅ Template rows: 75000
✅ patient_lengths mapped: 2 | bad_parse: 0
✅ test.csv columns: ['id', 'lead', 'fs', 'number_of_rows']
✅ pid_to_path: 0 | fs_map: 2
✅ glob indexed images: 8795
✅ YOLO loaded: /kaggle/input/ecg-final-models1/best.pt
✅ NEW loaded
✅ OLD loaded (resnet50)
🚀 Writing submission.csv (V40 FINAL)...


Processing Rows: 100%|██████████| 75000/75000 [00:15<00:00, 4727.88it/s]


✅ Done. submission.csv ready.


In [6]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.95 MB
🎉 Ready to Submit.
